
# Collecte OSM locale (extrait Geofabrik) — sans appels réseau

Ce notebook remplace la version Overpass API par une lecture **locale** de
l'extrait Algérie téléchargé sur Geofabrik (`algeria-latest.osm.pbf`).
Plus de rate-limit, plus de timeout, plus de dépendance à la disponibilité
d'un serveur public — tout est filtré en mémoire sur ta machine.

**Avant de lancer :**
1. Télécharge `algeria-latest.osm.pbf` depuis
   https://download.geofabrik.de/africa/algeria.html
2. `pip install osmium --break-system-packages`
3. Renseigne `PBF_PATH` ci-dessous vers le fichier téléchargé.

**Changement important par rapport à la version Overpass par wilaya :**
On ne peut plus garantir `wilaya_name` pour chaque élément comme avant
(là où on interrogeait wilaya par wilaya, on connaissait la wilaya à coup
sûr). Ici, tout le pays est lu en un seul passage, donc la wilaya est
retrouvée via le tag `addr:city`/`is_in` de l'élément (quand il existe) et
un matching flou **national** contre la table `communes`. C'est moins fiable
qu'avant, mais reste largement mieux que la situation initiale sur
`cabinet médical`/`laboratoire d'analyses` (qui n'avaient aucune wilaya).

Si tu veux plus tard une précision garantie, l'étape suivante serait
d'extraire les polygones de wilaya (relations `admin_level=4`) du même
fichier `.pbf` et de faire un vrai test point-dans-polygone (avec
`shapely`) — plus lourd à mettre en place, on peut le faire dans un second
temps si le matching texte s'avère insuffisant.


In [ ]:

import sqlite3
import unicodedata
import difflib
import osmium

DB_PATH = "data/dasec_prospection.db"
PBF_PATH = "algeria-latest.osm.pbf"   # <-- adapte le chemin si besoin

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print("Connecté à", DB_PATH)


## 1. Communes de référence (pour le matching wilaya/commune)

In [ ]:

def normaliser(texte: str) -> str:
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return texte.lower().strip()

# Index national : nom de commune normalisé -> (commune_id, commune_name, wilaya_name)
cursor = conn.execute("SELECT id, commune_name, wilaya_name FROM communes WHERE commune_name IS NOT NULL")
INDEX_COMMUNES = {}
for row in cursor.fetchall():
    INDEX_COMMUNES[normaliser(row["commune_name"])] = (row["id"], row["commune_name"], row["wilaya_name"])

print(f"{len(INDEX_COMMUNES)} communes indexées.")

def matcher_commune_nationale(texte_brut: str):
    """Matching flou national (pas de contexte wilaya préalable ici,
    contrairement à la version Overpass par wilaya). Retourne
    (commune_id, commune_name, wilaya_name) ou (None, None, None)."""
    if not texte_brut:
        return None, None, None
    cible = normaliser(texte_brut)
    if cible in INDEX_COMMUNES:
        return INDEX_COMMUNES[cible]
    proches = difflib.get_close_matches(cible, INDEX_COMMUNES.keys(), n=1, cutoff=0.8)
    if proches:
        return INDEX_COMMUNES[proches[0]]
    return None, None, None


## 2. Config des groupes à collecter (matchers de tags, équivalents aux filtres Overpass précédents)

In [ ]:

def classifier_hopital(nom: str) -> str:
    n = normaliser(nom)
    if "chu" in n:
        return "CHU"
    if "ehs" in n:
        return "EHS"
    if "eph" in n or "etablissement public hospitalier" in n:
        return "EPH"
    return "hôpital"

# Chaque groupe : secteur, sous_secteur (None si classifié après coup via
# `classifier`), et `match(tags: dict) -> bool`.
SOUS_SECTEURS_A_COLLECTER = [
    {
        "secteur": "santé", "sous_secteur": None, "classifier": classifier_hopital,
        "match": lambda t: t.get("amenity") == "hospital",
    },
    {
        "secteur": "santé", "sous_secteur": "clinique privée",
        "match": lambda t: t.get("healthcare") == "clinic" or t.get("amenity") == "clinic",
    },
    {
        "secteur": "santé", "sous_secteur": "laboratoire d'analyses",
        "match": lambda t: t.get("healthcare") == "laboratory",
    },
    {
        "secteur": "santé", "sous_secteur": "centre de transfusion sanguine",
        "match": lambda t: t.get("healthcare") == "blood_donation" or "transfusion" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "santé", "sous_secteur": "opticien",
        "match": lambda t: t.get("shop") == "optician",
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet dentaire",
        "match": lambda t: t.get("amenity") == "dentist" or t.get("healthcare") == "dentist",
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet médical",
        "match": lambda t: t.get("amenity") == "doctors" or t.get("healthcare") == "doctor",
    },
    {
        "secteur": "assurance", "sous_secteur": "assurance privée",
        "match": lambda t: t.get("office") == "insurance",
    },
    {
        "secteur": "assurance", "sous_secteur": "CNAS",
        "match": lambda t: "cnas" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "assurance", "sous_secteur": "CASNOS",
        "match": lambda t: "casnos" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet d'avocat",
        "match": lambda t: t.get("office") == "lawyer",
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet comptable",
        "match": lambda t: t.get("office") == "accountant",
    },
    {
        "secteur": "juridique", "sous_secteur": "notaire",
        "match": lambda t: t.get("office") == "notary",
    },
    {
        # Bruyant par nature -> à trier après import plutôt qu'à la collecte.
        "secteur": "industrie", "sous_secteur": "entreprise industrielle",
        "match": lambda t: t.get("building") == "industrial" and bool(t.get("name")),
    },
    {
        "secteur": "étatique", "sous_secteur": "mairie",
        "match": lambda t: t.get("amenity") == "townhall",
    },
    {
        "secteur": "étatique", "sous_secteur": "siège de wilaya",
        "match": lambda t: "wilaya de" in normaliser(t.get("name", "")),
    },
]

def trouver_groupe(tags: dict):
    for groupe in SOUS_SECTEURS_A_COLLECTER:
        if groupe["match"](tags):
            return groupe
    return None

print(f"{len(SOUS_SECTEURS_A_COLLECTER)} groupes configurés.")


## 3. Lecture du fichier .pbf (un seul passage, nœuds + chemins)

In [ ]:

class CollecteHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.resultats = []  # liste de dicts prêts à insérer

    def _extraire_commun(self, tags: dict, source_id: str, lat, lon, groupe) -> dict:
        return {
            "source_id": source_id,
            "secteur": groupe["secteur"],
            "sous_secteur": groupe["classifier"](tags.get("name")) if groupe.get("classifier") else groupe["sous_secteur"],
            "nom": tags.get("name"),
            "telephone": tags.get("contact:phone") or tags.get("phone"),
            "email": tags.get("contact:email") or tags.get("email"),
            "site_web": tags.get("contact:website") or tags.get("website"),
            "facebook": tags.get("contact:facebook"),
            "latitude": lat,
            "longitude": lon,
            "commune_brute": tags.get("addr:city") or tags.get("addr:municipality") or tags.get("is_in"),
            "adresse": tags.get("addr:full") or tags.get("addr:street"),
        }

    def node(self, n):
        tags = {tag.k: tag.v for tag in n.tags}
        if not tags:
            return
        groupe = trouver_groupe(tags)
        if groupe is None:
            return
        if not n.location.valid():
            return
        self.resultats.append(self._extraire_commun(tags, f"node/{n.id}", n.location.lat, n.location.lon, groupe))

    def way(self, w):
        tags = {tag.k: tag.v for tag in w.tags}
        if not tags:
            return
        groupe = trouver_groupe(tags)
        if groupe is None:
            return
        lats, lons = [], []
        for node_ref in w.nodes:
            if node_ref.location.valid():
                lats.append(node_ref.location.lat)
                lons.append(node_ref.location.lon)
        if not lats:
            return
        lat = sum(lats) / len(lats)
        lon = sum(lons) / len(lons)
        self.resultats.append(self._extraire_commun(tags, f"way/{w.id}", lat, lon, groupe))


print("Lecture du fichier .pbf en cours (peut prendre quelques minutes selon la taille)...")

idx = osmium.index.create_map("sparse_mem_array")
lh = osmium.NodeLocationsForWays(idx)
lh.ignore_errors()  # ignore les ways dont un nœud référencé serait absent de l'extrait

handler = CollecteHandler()
osmium.apply(osmium.io.Reader(PBF_PATH), lh, handler)

print(f"{len(handler.resultats)} éléments correspondant à un groupe trouvés.")


## 4. Insertion dédupliquée en base

In [ ]:

def deja_importe(source_id: str) -> bool:
    row = conn.execute("SELECT id FROM entreprises WHERE source_id = ?", (source_id,)).fetchone()
    return row is not None

def inserer_etablissement(champs: dict) -> bool:
    if deja_importe(champs["source_id"]):
        return False

    nom = champs["nom"] or f"{champs['sous_secteur']} (sans nom)"
    commune_id, commune_name, wilaya_name = matcher_commune_nationale(champs["commune_brute"])

    valeurs = (
        nom, champs["secteur"], champs["sous_secteur"], champs["telephone"], champs["email"],
        champs["site_web"], champs["facebook"], champs["latitude"], champs["longitude"],
        commune_id, commune_name or champs["commune_brute"], wilaya_name,
        champs["adresse"], champs["source_id"],
    )

    try:
        conn.execute("""
            INSERT INTO entreprises (
                nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                latitude, longitude, commune_id, commune_brute, wilaya_name,
                adresse, source, source_id
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'osm', ?)
        """, valeurs)
        return True
    except sqlite3.IntegrityError:
        source_unique = f"osm#{champs['source_id']}"
        try:
            conn.execute("""
                INSERT INTO entreprises (
                    nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                    latitude, longitude, commune_id, commune_brute, wilaya_name,
                    adresse, source, source_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, valeurs[:-1] + (source_unique, champs["source_id"]))
            return True
        except sqlite3.IntegrityError as e:
            print(f"  [ignoré définitivement] {champs['source_id']} : {e}")
            return False


inseres = 0
sans_wilaya = 0

for champs in handler.resultats:
    if inserer_etablissement(champs):
        inseres += 1
        if not champs.get("commune_brute"):
            sans_wilaya += 1

conn.commit()
print(f"{inseres} nouveaux établissements insérés.")
print(f"Dont {sans_wilaya} sans tag commune exploitable (wilaya_name restera NULL pour ceux-là).")


## 5. Vérification post-import

In [ ]:

cursor = conn.execute("""
    SELECT secteur, sous_secteur,
           COUNT(DISTINCT wilaya_name) as nb_wilayas,
           SUM(CASE WHEN wilaya_name IS NULL THEN 1 ELSE 0 END) as sans_wilaya,
           COUNT(*) as total
    FROM entreprises
    WHERE source LIKE 'osm%'
    GROUP BY secteur, sous_secteur
    ORDER BY nb_wilayas ASC
""")
for row in cursor.fetchall():
    print(f"{row['secteur']:12} {row['sous_secteur'] or '—':30} {row['nb_wilayas']:3} wilayas   "
          f"{row['sans_wilaya']:4} sans wilaya   {row['total']:5} total")

conn.close()
